# 🫀 Multi-Modal Heart Disease Prediction System (100% XGBoost Architecture)
### Combining Clinical Tabular Patient Data (`heart.csv`) & 12-Lead ECG Waveform Images (`xgboost_ecg_model.json`)

This end-to-end multi-modal pipeline uses **XGBoost across all components**:
1. **Tabular Clinical Classifier**: Trained with **XGBoost Classifier** on 13 patient health attributes (`age`, `chol`, `bp`, `cp`, etc.) from `heart.csv`
2. **ECG Vision Model**: Pre-trained **XGBoost 100-Tree Model** loaded from `xgboost_ecg_model.json` for 12-lead ECG image analysis
3. **Unified Multi-Modal XGBoost Decision Engine**: Fuses both XGBoost outputs into a single diagnostic verdict: **AFFECTED (Heart Disease)** or **HEALTHY**.

## 1. Install & Import Dependencies

In [ ]:
%pip install xgboost scikit-learn pandas numpy matplotlib pillow -q

import os
import glob
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, classification_report

print(f"Using XGBoost Version: {xgb.__version__}")

## 2. Train Tabular Clinical Classifier with XGBoost (`heart.csv`)

In [ ]:
# Candidate search paths for heart.csv
candidate_tabular_paths = [
    'heart.csv',
    '/content/heart.csv',
    'backend/app/models/heart_disease/heart.csv',
    'app/models/heart_disease/heart.csv',
    '/content/Mini-Project/backend/app/models/heart_disease/heart.csv'
]

tabular_path = next((p for p in candidate_tabular_paths if os.path.exists(p)), None)

if not tabular_path:
    url = 'https://raw.githubusercontent.com/dileep-lingamallu/Heart-Disease-Prediction-Dataset/master/heart.csv'
    print("Downloading heart.csv to session...")
    try:
        urllib.request.urlretrieve(url, 'heart.csv')
        tabular_path = 'heart.csv'
    except Exception:
        tabular_path = url

print(f"Loading Tabular Dataset: {tabular_path}")
tabular_df = pd.read_csv(tabular_path)
print(f"Loaded {len(tabular_df)} patient records with {len(tabular_df.columns)} columns.")

# Feature-target split
X_tab = tabular_df.drop(columns=['target'])
y_tab = tabular_df['target']

X_tab_train, X_tab_test, y_tab_train, y_tab_test = train_test_split(
    X_tab, y_tab, test_size=0.2, random_state=42, stratify=y_tab
)

# Train Clinical Tabular XGBoost Classifier
tabular_xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)
tabular_xgb_model.fit(X_tab_train, y_tab_train)

y_tab_pred = tabular_xgb_model.predict(X_tab_test)
y_tab_proba = tabular_xgb_model.predict_proba(X_tab_test)[:, 1]

print("\n--- XGBoost Tabular Classifier Performance ---")
print(f"Accuracy : {accuracy_score(y_tab_test, y_tab_pred) * 100:.2f}%")
print(f"Precision: {precision_score(y_tab_test, y_tab_pred) * 100:.2f}%")
print(f"Recall   : {recall_score(y_tab_test, y_tab_pred) * 100:.2f}%")
print(f"ROC-AUC  : {roc_auc_score(y_tab_test, y_tab_proba):.4f}")

# Save trained tabular XGBoost model
tabular_xgb_model.save_model('xgboost_tabular_model.json')
print("\nSaved tabular model to xgboost_tabular_model.json successfully!")

## 3. Load Pre-Trained XGBoost ECG Vision Model (`xgboost_ecg_model.json`)

The 100-tree XGBoost vision model operates on $128 \times 128 \times 3 = 49,152$ flattened ECG waveform pixels.

In [ ]:
# Candidate search paths for xgboost_ecg_model.json
candidate_xgb_paths = [
    'xgboost_ecg_model.json',
    '/content/xgboost_ecg_model.json',
    'backend/app/models/heart_disease/xgboost_ecg_model.json',
    'app/models/heart_disease/xgboost_ecg_model.json',
    '/content/Mini-Project/backend/app/models/heart_disease/xgboost_ecg_model.json'
]

xgb_model_path = next((p for p in candidate_xgb_paths if os.path.exists(p)), None)

if xgb_model_path:
    ecg_booster = xgb.Booster()
    ecg_booster.load_model(xgb_model_path)
    print(f"✅ Successfully loaded trained XGBoost ECG model from: {xgb_model_path}")
    print(f"🌲 Number of boosted trees: {ecg_booster.num_boosted_rounds()}")
else:
    print("⚠️ xgboost_ecg_model.json not found in search paths. Please upload it to your session.")

## 4. Multi-Modal Unified XGBoost Prediction Engine

In [ ]:
def predict_patient_heart_status(tabular_input, ecg_image_path=None, weight_tabular=0.5, weight_ecg=0.5):
    """
    100% XGBoost Multi-Modal Diagnosis Engine combining:
    1. Clinical Tabular XGBoost Model (13 patient vitals)
    2. ECG Strip XGBoost Vision Model (49,152 image features)
    """
    # 1. Clinical Tabular XGBoost Inference
    tab_features = np.asarray(tabular_input).reshape(1, -1)
    tab_prob = float(tabular_xgb_model.predict_proba(tab_features)[0][1])
    
    # 2. ECG Image XGBoost Inference
    ecg_prob = None
    if ecg_image_path and os.path.exists(ecg_image_path):
        img = Image.open(ecg_image_path).convert('RGB').resize((128, 128))
        feat = np.array(img, dtype=np.float32).flatten() / 255.0
        dmat = xgb.DMatrix(feat.reshape(1, -1))
        ecg_prob = float(ecg_booster.predict(dmat)[0])
    else:
        print("Notice: No ECG image supplied. Relying solely on Tabular XGBoost model.")
        ecg_prob = tab_prob
        
    # 3. Fused Dual-XGBoost Risk Score
    fused_risk = (weight_tabular * tab_prob) + (weight_ecg * ecg_prob)
    is_affected = fused_risk >= 0.50
    
    print("=" * 65)
    print("🫀 100% XGBOOST MULTI-MODAL HEART DISEASE REPORT")
    print("=" * 65)
    print(f"• Clinical Tabular XGBoost Risk : {tab_prob * 100:.2f}%")
    print(f"• ECG Vision XGBoost Risk       : {ecg_prob * 100:.2f}%")
    print(f"• Combined Dual-XGBoost Risk    : {fused_risk * 100:.2f}%")
    print("-" * 65)
    
    if is_affected:
        print("🚨 FINAL STATUS : AFFECTED (Cardiac Abnormality / Heart Disease Detected)")
        print("⚕️ Recommendation: High cardiac risk. Immediate cardiologist referral advised.")
    else:
        print("✅ FINAL STATUS : HEALTHY (Normal Cardiac Metrics)")
        print("⚕️ Recommendation: Normal sinus rhythm & clinical parameters.")
    print("=" * 65)
    
    return {
        "is_affected": bool(is_affected),
        "fused_risk_score": round(fused_risk, 4),
        "tabular_xgb_risk": round(tab_prob, 4),
        "ecg_xgb_risk": round(ecg_prob, 4)
    }

## 5. Live Testing with Dual-XGBoost Pipeline

In [ ]:
# Find sample test images from local or Colab directories
normal_samples = glob.glob('**/Normal(1)*.jpg', recursive=True) + glob.glob('**/Normal*.jpg', recursive=True)
mi_samples = glob.glob('**/MI(1)*.jpg', recursive=True) + glob.glob('**/MI*.jpg', recursive=True)

sample_normal = normal_samples[0] if normal_samples else None
sample_mi = mi_samples[0] if mi_samples else None

# Test Case 1: High-Risk Patient (Angina, ST-depression, High BP) + Myocardial Infarction ECG Image
print("TEST CASE 1: Affected Patient")
patient_affected = (70, 1, 0, 145, 174, 0, 1, 125, 1, 2.6, 0, 0, 3)
res_affected = predict_patient_heart_status(patient_affected, sample_mi)

print("\n")
# Test Case 2: Healthy Patient (Younger, Normal BP & Chol, High Max HR) + Normal ECG Image
print("TEST CASE 2: Healthy Patient")
patient_healthy = (38, 0, 2, 115, 180, 0, 0, 172, 0, 0.0, 2, 0, 2)
res_healthy = predict_patient_heart_status(patient_healthy, sample_normal)